## MLS WS 2025/26 Exercise 3c: NIDS Model Performance (20 Points)
*Adapted from an exercise created by Dennis Eisermann*

In this exercise, you analyze the classification model for the [CIC-IDS-2017](https://www.unb.ca/cic/datasets/ids-2017.html) cybersecurity dataset from the previous exercise. Feel free to reuse parts from the solution of the last exercise notebook for this submission. In the course of this exercise, we will improve the results with a new architecture.

### Setup

In [ ]:
%pip install torchmetrics
%pip install torch --index-url https://download.pytorch.org/whl/cu126

Use the train_data.pt, val_data.pt, and test_data.pt from the last exercise notebook. Also download small_ids.pth as starting model.

### Task 1 – Model Evaluation (10 Points)
There are some [pitfalls](https://machinelearningmastery.com/tour-of-evaluation-metrics-for-imbalanced-classification/) to evaluate datasets with imbalanced data. The goal of this task is to perform a more granular analysis in comparison to the last exercise notebook.

1. Load the training, validation, and test datasets from the last notebook. (1 Point)

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.load("train_data.pt", weights_only=False) 
y_tensor = torch.load("train_labels.pt", weights_only=False) 
X_val_tensor = torch.load("val_data.pt", weights_only=False) 
y_val_tensor = torch.load("val_labels.pt", weights_only=False) 
X_test_tensor = torch.load("test_data.pt", weights_only=False)
y_test_tensor = torch.load("test_labels.pt", weights_only=False)

test_data_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=256)
train_loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=128)


2. Load the small IDS model from the last exercise notebook. (1 Point)

In [ ]:
import torch.nn as nn

class SmallIDSNet(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
import torch

PATH = "ids_model.pth"
small_ids_network = SmallIDSNet(68,1)
small_ids_network.load_state_dict(torch.load(PATH))
small_ids_network.eval()
small_ids_network.to('cuda')

3. Classify the test dataset with the `SmallIDSNet` model and inspect metrics for each of the classes. As metrics calculate Accuracy, Precision, and Recall. Write a function to repeat the calculation of metrics as needed. (5 Points)

In [ ]:
import torch
from torchmetrics.functional import accuracy, precision, recall

classes = ['Benign', 'Malicious']

def evaluate(model, test_loader, class_names=classes):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            if torch.cuda.is_available():
                inputs = inputs.to('cuda')
                labels = labels.to('cuda')
            outputs = model(inputs)
            preds = (outputs > 0)
            all_preds.extend(preds.cpu().numpy().astype(int))
            all_labels.extend(labels.cpu().numpy())
    
    all_preds_tensor = torch.tensor(all_preds)
    all_labels_tensor = torch.tensor(all_labels)
    
    acc = accuracy(all_preds_tensor.flatten(), all_labels_tensor.flatten(), task='binary', num_classes=len(class_names))
    prec = precision(all_preds_tensor.flatten(), all_labels_tensor.flatten(), task='binary', num_classes=len(class_names))
    rec = recall(all_preds_tensor.flatten(), all_labels_tensor.flatten(), task='binary', num_classes=len(class_names))

    return (acc, prec, rec)


acc, prec, rec = evaluate(small_ids_network, test_data_loader, class_names=classes)

print(f"Accuracy: {acc.item():.4f}")
print(f"Precision: {prec.item():.4f}")
print(f"Recall: {rec.item():.4f}")

4. Which new insides could you gain from this analysis? (3 Points)

Precision and Recall are quite low. This indicates that the model is not very good at identifying the different types of attacks. The model may be biased towards the majority class (Benign), leading to poor performance on minority classes. This suggests that the model may not be effectively capturing the characteristics of the different attack types.

### Task 2 – Model Adjustment (11 Points)
Your goal is to improve the model architecture. We recommend the [Deep Learning Tuning Playbook](https://github.com/google-research/tuning_playbook?tab=readme-ov-file#choosing-the-model-architecture) for a quickstart into the topic.

1. Design an improved model architecture with modifications that are likely to improve the classification quality. (3 Points)

In [ ]:
import torch.nn as nn

class MediumIDSNet(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        return self.net(x)

2. Write a function that encapsulates the training process. Reuse the code from the last exercise notebook. (1 Points)

In [ ]:
import torch.optim
import numpy

def train(model, loader, epochs, learning_rate):
    model.train()
    if torch.cuda.is_available():
        model.to('cuda')

    optimizer = torch.optim.NAdam(model.parameters(), lr=learning_rate)

    criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(1, epochs+1):
        avg_epoch_losses = []
        losses, accuracies = [], []
        print(f"Epoch {epoch} Start...")
        for i, data in enumerate(loader, 0):
            inputs, labels = data
            
            if torch.cuda.is_available():
                inputs = inputs.to('cuda')
                labels = labels.to('cuda')

            optimizer.zero_grad()
            outputs = model(inputs)
            
            loss = criterion(outputs.flatten(), labels)
            # if i % 160 == 0:
            #     print()
            # # print("Batch: {:3}, Loss {:.4f}: ".format(i, loss), end=" ")
            # print('.', end="")

            loss.backward()
            optimizer.step()

            local_loss = loss.cpu()
            losses.append(local_loss.detach().numpy())

        avg_epoch_loss = numpy.asarray(losses).mean()
        avg_epoch_losses.append(avg_epoch_loss)
        print("Epoch {} end:, average loss: {:.04f}".format(epoch, avg_epoch_loss))
    
    # print(f"Epoch {epoch:02d}")
    # print(f"Train Loss: {avg_loss:.4f}")
    # print(f"Validation Acc: {val_accuracy:.4f}")
    # print(f"Train Acc:{training_accuracy:.4f}")
    print()

3. Train and validate the medium size model with your functions. Adjust epochs and learning rate at your discretion. (0 Points)

In [20]:
torch.set_float32_matmul_precision('high') # Depends on hardware
print("Medium Model")
medium_model = MediumIDSNet(68, 1)
train(medium_model, train_loader, 2, 1e-4)
medium_results = evaluate(medium_model, test_data_loader, class_names=classes)
print(f"Accuracy: {medium_results[0]:.4f}")
print(f"Precision: {medium_results[1]:.4f}")
print(f"Recall: {medium_results[2]:.4f}")

Medium Model
Epoch 1 Start...

Epoch 1 end:, average loss: 0.0466
Epoch 2 Start...

Epoch 2 end:, average loss: 0.0952

Accuracy: 0.1969
Precision: 0.1969
Recall: 1.0000


4. Discuss how the architecture influences the result. (2 Points)

A more complex architecture with additional layers and neurons can capture more intricate patterns in the data, leading to improved classification performance. However, it also increases the risk of overfitting, especially with limited data. Regularization techniques such as dropout and batch normalization can help mitigate this risk.

5. Discuss how the epoch influences the result. (2 Points)

The number of epochs is proportional to the training time and the training error. Once the training error stagnates additional epochs will not improve the model performance. However, too few epochs can lead to underfitting, where the model has not learned enough from the data. The number epochs has an inverse relation to the learning rate. A higher learning rate requires fewer epochs to converge, while a lower learning rate may require more epochs.

6. Discuss how the learning rate influences the result. (3 Points)

the learning rate influences how quickly the model converges to a solution. A high learning rate can lead to faster convergence but may overshoot the optimal solution, resulting in no learning at all. Conversely, a low learning rate allows for more precise adjustments to the model weights, but it requires a larger number of epochs to converge, increasing training time. 